<a href="https://colab.research.google.com/github/dmahapatra/smartindustry-metropolia-deb/blob/main/rag_application_metropolia_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install langchain langchain-community langchain-huggingface langchain-text-splitters \
faiss-cpu sentence-transformers transformers accelerate bitsandbytes pypdf requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests

In [2]:
import os
import requests
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [8]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/dmahapatra/smartindustry-metropolia-deb/main/FinOps_Documentation.pdf"
LOCAL_FILE = "/content/finops_doc.pdf" # Changed extension to .pdf

r = requests.get(GITHUB_RAW_URL)
r.raise_for_status()

with open(LOCAL_FILE, "wb") as f:
    f.write(r.content)

print("Downloaded:", LOCAL_FILE, "Size:", os.path.getsize(LOCAL_FILE), "bytes")

Downloaded: /content/finops_doc.pdf Size: 113622 bytes


**Load the document (TXT/MD or PDF)**

In [10]:
file_path = LOCAL_FILE
suffix = Path(file_path).suffix.lower()

if suffix in [".txt", ".md"]:
    loader = TextLoader(file_path, encoding="utf-8")
    documents = loader.load()
elif suffix == ".pdf":
    loader = PyPDFLoader(file_path)
    documents = loader.load()
else:
    raise ValueError(f"Unsupported file type: {suffix}. Use .txt, .md, or .pdf")

print("Loaded documents:", len(documents))
print("Sample preview:\n", documents[0].page_content[:500])

Loaded documents: 8
Sample preview:
 Financial Operations (FinOps) in Cloud: Integrating
financial management practices in cloud operations.
Date: 3rd May 2024
Author
Maahir Azlaan
Abstract
FinOps, or Financial Operations, is a strategic framework designed to optimize cloud financial
management by fostering collaboration between finance, engineering, and business teams. As
organizations increasingly rely on cloud services, effective management of cloud expenditures becomes
critical. This document explores the foundational elements 


**Split the document into chunks**

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("First chunk:\n", chunks[0].page_content[:500])

Total chunks: 30
First chunk:
 Financial Operations (FinOps) in Cloud: Integrating
financial management practices in cloud operations.
Date: 3rd May 2024
Author
Maahir Azlaan
Abstract
FinOps, or Financial Operations, is a strategic framework designed to optimize cloud financial
management by fostering collaboration between finance, engineering, and business teams. As
organizations increasingly rely on cloud services, effective management of cloud expenditures becomes
critical. This document explores the foundational elements 


**Create embeddings (open-source model)**

In [12]:
embedding_model_name = "BAAI/bge-small-en-v1.5"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cpu"},   # Colab CPU-safe; if GPU available, can use "cuda"
    encode_kwargs={"normalize_embeddings": True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Build the FAISS vector store**

In [13]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Vector store created successfully.")

Vector store created successfully.


**Load an open-source LLM (LangChain + Hugging Face)**

In [14]:
llm_model_name = "google/flan-t5-base"  # You can also try flan-t5-large if runtime allows

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

hf_pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=hf_pipe)
print("LLM loaded:", llm_model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


LLM loaded: google/flan-t5-base


**Create a simple RAG prompt + answer function**

In [17]:
def answer_question(question: str):
    # 1) Retrieve relevant chunks
    relevant_docs = retriever.invoke(question) # Changed get_relevant_documents to invoke

    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # 2) Prompt
    prompt = f"""
You are a FinOps assistant. Answer the user's question ONLY using the provided context.
If the answer is not in the context, say: "I could not find that in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # 3) Generate answer
    response = llm.invoke(prompt)

    # 4) Return answer + sources (for debugging/transparency)
    return {
        "question": question,
        "answer": response,
        "retrieved_chunks": [doc.page_content[:400] for doc in relevant_docs]
    }

**Ask a question (user input)**

In [18]:
user_question = input("Enter your FinOps question: ")

result = answer_question(user_question)

print("\n=== Answer ===")
print(result["answer"])

print("\n=== Retrieved Chunks (preview) ===")
for i, chunk in enumerate(result["retrieved_chunks"], 1):
    print(f"\nChunk {i}:\n{chunk}\n{'-'*60}")

Enter your FinOps question: what is finops?


Token indices sequence length is longer than the specified maximum sequence length for this model (615 > 512). Running this sequence through the model will result in indexing errors



=== Answer ===
FinOps, short for Financial Operations, is a set of practices and tools designed to manage cloud financial management and optimize cloud spending.

=== Retrieved Chunks (preview) ===

Chunk 1:
A. Definition of FinOps
FinOps, short for Financial Operations, is a set of practices and tools designed to manage cloud financial
management and optimize cloud spending. It brings together finance, technology, and business teams to
ensure that cloud costs are aligned with business goals and that financial accountability is maintained
throughout the cloud lifecycle.
B. Importance of FinOps in Clou
------------------------------------------------------------

Chunk 2:
A. Recap of the Importance of FinOps
FinOps is essential for managing cloud financials effectively, ensuring that organizations optimize their
cloud spending while aligning costs with business objectives. By fostering collaboration, accountability,
and continuous improvement, FinOps enables better financial decision-m